# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
([FEA-026: identity and transactions](../../../mbse-basis/docs/features/FEA-026.md)). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. An object is a singleton, owned, or counted: a counted
object has no identity beyond the document that writes it, which numbers it.

In [ ]:
import json

from mbse.Schemas.Framework import Builders as B, Paths, Plain, Schemas as S, Validators
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import listed, raises, view


def wts(txn):
    """The directory of definitions, each owning its clauses: a singleton wts.Features owns each wts.Definition through
    wts.Defines, by a name its pattern gives, and a definition each wts.Clause through wts.HasClause."""
    S = txn.Schemas
    return txn.uid("wts/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Directory": S.OfObject, "Definition": S.OfObject, "Clause": S.OfObject, "Defines": S.OfRelation,
                  "HasClause": S.OfRelation})
        .spec([
            lambda _: _.Directory.name("wts.Directory").ref(True).singleton("wts.Features").adjacencies([
                lambda t: t.name("dir").relation([_.Defines]).me("directory")]),
            lambda _: _.Definition.name("wts.Definition").ref(True).properties([
                lambda t: t.name("title").schema([S.OfStr]),
            ]).adjacencies([
                lambda t: t.name("def").relation([_.Defines]).me("defn").owner(True),
                lambda t: t.name("clauses").relation([_.HasClause]).me("defn"),
            ]),
            lambda _: _.Clause.name("wts.Clause").ref(True).adjacencies([
                lambda t: t.relation([_.HasClause]).me("clause").owner(True)]),
            lambda _: (
                _.Defines
                .name("wts.Defines")
                .links([
                    lambda t: t.name("directory"),
                    lambda t: t.name("defn"),
                ])
                .properties([lambda t: t.name("name").schema([S.OfStr]).pattern(r"DEF-(\d+\d\d)")])
                .uniques([["directory", "name"], ["defn"]])
            ),
            lambda _: (
                _.HasClause
                .name("wts.HasClause")
                .links([
                    lambda t: t.name("defn"),
                    lambda t: t.name("clause"),
                ])
                .properties([lambda t: t.name("name").schema([S.OfStr])])
                .uniques([["defn", "name"], ["clause"]])
            ),
        ])]))])


store = B.Store()
Text = store.Schemas.OfStr.as_schema()
store.transaction(wts)
Directory, Definition, Clause = store.registered("wts.Directory"), store.registered("wts.Definition"), store.registered("wts.Clause")
Defines, HasClause = store.registered("wts.Defines"), store.registered("wts.HasClause")

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
with store.transaction():
    assert Definition.adjacencies["def"].owner is True and Directory.adjacencies["dir"].owner is None
    assert [s.validate() for s in (Directory, Definition, Clause, Defines, HasClause)] == [[]] * 5


    # An object schema owned through a relation, each validated: the relation's links and unique clauses decide.
    assert store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([lambda t: t.name("up").relation([
        store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("up"), lambda t: t.name("down")]).uniques([["up"], ["down"]])),
    ]).me("down").owner(True)])).validate() == []  # keyless: the owner alone determines the owned object
    assert store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([lambda t: t.name("up").relation([
        store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("up"), lambda t: t.name("down")]).uniques([["down"]])),
    ]).me("down").owner(True)])).validate() == [
        "adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that the owned object "
        "determines its owner"]
    assert store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([lambda t: t.name("up").relation([
        store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("up"), lambda t: t.name("down")]).uniques([["up"]])),
    ]).me("down").owner(True)])).validate() == [
        "adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that the owner and the keys "
        "determine the owned object"]
    assert store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([lambda t: t.name("up").relation([
        store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("up"), lambda t: t.name("down"), lambda t: t.name("side")])
                                 .uniques([["up"], ["down"]])),
    ]).me("down").owner(True)])).validate() == [
        "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]
    with raises(TypeError, match="property 'owner' holds bool, got str"):
        store.Schemas.OfObject(lambda o: o.adjacencies([lambda t: t.relation([Defines]).me("defn").owner("yes")]))

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
with store.transaction():
    name = Defines.entry.properties["name"]
    assert name.pattern == r"DEF-(\d+\d\d)" and name.name_for(7) == "DEF-007" and name.name_for(1234) == "DEF-1234"
    assert (name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")) == (693, None, None)
    assert S.OfProperty.Data("k", Text).number_of("DEF-693") is None  # no pattern
    bad = store.Schemas.OfRelation(lambda r: (
        r
        .links([lambda t: t.name("a"), lambda t: t.name("b")])
        .properties([
            lambda t: t.name("k").schema([Text]).pattern("DEF-.*"),
            lambda t: t.name("n").schema([store.Schemas.OfInt]).pattern(r"(\d+)"),
        ])
    ))
    assert bad.validate() == [
        "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
        "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]

## IDN-03 · Every store holds the path schemas; the members of its own relations are owned

In [ ]:
for name in ("Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"):
    assert store.registered(name).validate() == [] and store.framework(store.registered(name))
assert list(store.Paths.OfRoot.as_schema().properties) == ["singleton", "next"]
assert list(store.Paths.OfEntry.as_schema().properties) == ["i", "o", "properties", "next"]
assert [b.name for b in store.Paths.Next.as_schema().branches] == ["entry", "target"]
other = B.Store()
assert other.Paths.OfRoot.as_schema() is not store.Paths.OfRoot.as_schema()  # each store's own
assert "instances" not in store.Schemas.OfObject.as_schema().adjacencies  # a schema holds no instances
owning = [(a.relation.name, a.me) for a in store.Schemas.OfProperty.as_schema().anonymous if a.owner]
assert owning == [("Schemas.HasProperty", "property")]
assert all(a.owner for a in store.Schemas.OfObject.as_schema().anonymous if a.relation.name == "Schemas.Defines")

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
features = store.Instances.wts.Features
assert features is store.singleton("wts.Features")
with store.transaction():
    store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.name("DEF-693").defn(lambda d: (
        d
        .title("Has 88 keys")
        .clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x)))
    )))))
definition = features.dir({"name": "DEF-693"})
clause = definition.clauses({"name": "C-1"})
to_plain = Plain.ToPlain(store)
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(definition)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"target": {}}}}}
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(clause)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"entry": {
        "i": "clauses", "o": "", "properties": {"name": "C-1"}, "next": {"target": {}}}}}}}  # an anonymous owner
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(features)) == {"singleton": "wts.Features",
                                                                         "next": {"target": {}}}
assert Paths.same(Paths.of(clause), Paths.of(clause)) and not Paths.same(Paths.of(clause), Paths.of(definition))
assert Paths.same(None, None) and not Paths.same(Paths.of(clause), None)
with store.transaction():
    store.wts.Definition(definition, lambda d: d.clauses(lambda a: a.remove(lambda e: e.clause(clause))))
with raises(LookupError, match="a 'wts.Clause' has no owner: its path is not known"):
    Paths.of(clause)  # its owning entry removed


def twice(txn):
    """wts.Twice, owned through wts.Holds by another wts.Twice."""
    S = txn.Schemas
    return txn.uid("twice/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Twice": S.OfObject, "Holds": S.OfRelation})
        .spec([
            lambda _: _.Twice.name("wts.Twice").ref(True).adjacencies([
                lambda t: t.name("up").relation([_.Holds]).me("b").owner(True),
                lambda t: t.name("down").relation([_.Holds]).me("a"),
            ]),
            lambda _: _.Holds.name("wts.Holds").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([["a"], ["b"]]),
        ])]))])


store.transaction(twice)
with store.transaction():  # within one transaction, objects are given their owners when it ends: here, each other
    first, second = store.wts.Twice(lambda t: t), store.wts.Twice(lambda t: t)
    store.wts.Twice(first, lambda t: t.up(lambda a: a.insert(lambda e: e.a(second))))
    store.wts.Twice(second, lambda t: t.up(lambda a: a.insert(lambda e: e.a(first))))
with raises(LookupError, match="an ownership cycle: no singleton owns the object"):
    Paths.of(first)


def splits(txn):
    """wts.Split, which wts.Features may own two ways, through wts.Lefts and wts.Rights."""
    S = txn.Schemas
    return txn.uid("split/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Lefts": S.OfRelation, "Rights": S.OfRelation, "Directory": S.OfObject, "Split": S.OfObject})
        .spec([  # named: each end's adjacency links it
            lambda _: _.Lefts.name("wts.Lefts").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([["a"], ["b"]]),
            lambda _: _.Rights.name("wts.Rights").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([["a"], ["b"]]),
            lambda _: _.Directory.name("wts.Directory").adjacencies([lambda t: t.relation([_.Lefts]).me("a"),
                                                                    lambda t: t.name("rights").relation([_.Rights]).me("a")]),
            lambda _: _.Split.name("wts.Split").ref(True).adjacencies([
                lambda t: t.name("left").relation([_.Lefts]).me("b").owner(True),
                lambda t: t.name("right").relation([_.Rights]).me("b").owner(True),
            ]),
        ])]))])


store.transaction(splits)
with store.transaction():
    split = store.wts.Split(lambda s: s.left(lambda a: a.insert(lambda e: e.a(features))))
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(split)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "", "o": "left", "properties": {}, "next": {"target": {}}}}}  # the owner's adjacency anonymous
with store.transaction():
    store.wts.Split(split, lambda s: s.right(lambda a: a.insert(lambda e: e.a(features))))
with raises(LookupError, match="a 'wts.Split' has more than one owner: its path is not known"):
    Paths.of(split)
with raises(ReadOnlyError, match="'as_identity' is not a property or adjacency of 'wts.Twice'"):
    with store.transaction():
        store.wts.Twice(first, lambda t: t.as_identity("twice-9"))  # a program writes no identity

## IDN-05 · An object built without an owner is given the one its schema allows: a singleton's, its keys from their naming patterns; else refused. Removing its owning entry deletes it, and what it owns

In [ ]:
with store.transaction():
    store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.defn(lambda d: d.title("nameless")))))  # no key
    made = store.wts.Definition(lambda d: d.title("made alone"))  # its owner the one singleton that can be, Features
assert features.dir({"name": "DEF-694"}) is made  # its key the next its pattern gives, past every number held
with store.transaction():
    defn = store.wts.Definition(lambda d: d)
assert defn is features.dir({"name": "DEF-695"})
with raises(ValueError, match="a 'wts.Clause' needs an owner through 'wts.HasClause': give it one, since no singleton can be"):
    with store.transaction():
        store.wts.Clause(lambda c: c)  # a definition owns it, and none is a singleton
with raises(ValueError, match="a 'wts.Split' needs an owner through 'left', 'right': give it one, since more than one "
                              "singleton can be"):
    with store.transaction():
        store.wts.Split(lambda s: s)


def notes(txn):
    """wts.Note, owned by wts.Features through wts.Notes, by a name."""
    S = txn.Schemas
    return txn.uid("notes/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Notes": S.OfRelation, "Directory": S.OfObject, "Note": S.OfObject})
        .spec([
            lambda _: (
                _.Notes
                .name("wts.Notes")
                .links([lambda t: t.name("directory"), lambda t: t.name("note")])
                .properties([lambda t: t.name("name").schema([S.OfStr])])  # a key with no naming pattern
                .uniques([["directory", "name"], ["note"]])
            ),
            lambda _: _.Directory.name("wts.Directory").adjacencies([lambda t: t.name("notes").relation([_.Notes]).me("directory")]),
            lambda _: _.Note.name("wts.Note").ref(True).adjacencies([
                lambda t: t.relation([_.Notes]).me("note").owner(True),
            ]),
        ])]))])


store.transaction(notes)
with raises(ValueError, match="a 'wts.Note' needs an owner: the key 'name' of 'wts.Notes' has no naming pattern to give "
                              "it a value"):
    with store.transaction():
        store.wts.Note(lambda n: n)
with store.transaction():
    store.wts.Directory(features, lambda f: f.notes(lambda a: a.insert(lambda e: e.name("first").note(lambda n: n))))
assert features.notes({"name": "first"}).as_schema() is store.registered("wts.Note")

# Removing an owning entry deletes the owned object: every entry it is in goes, and so, in turn, what it owned.
with store.transaction():
    gone = store.wts.Definition(lambda d: d.title("gone").clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x))))
its = gone.clauses({"name": "C-1"})
with store.transaction():
    store.wts.Directory(features, lambda f: f.dir(lambda a: a.remove(lambda e: e.defn(gone))))
assert len(gone.clauses) == 0 and all(e.defn is not gone for e in features.dir)
with raises(LookupError, match="a 'wts.Clause' has no owner: its path is not known"):
    Paths.of(its)  # deleted with the definition that owned it

# Nothing is owned by a counted object: an object neither a singleton nor owned.


def kept(txn):
    """wts.Loose, counted, which would own a wts.Kept through wts.Keeps."""
    S = txn.Schemas
    return txn.uid("kept/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Loose": S.OfObject, "Keeps": S.OfRelation, "Kept": S.OfObject})
        .spec([
            lambda _: _.Loose.name("wts.Loose").ref(True).adjacencies([lambda t: t.name("keeps").relation([_.Keeps]).me("keeper")]),
            lambda _: _.Keeps.name("wts.Keeps").links([lambda t: t.name("keeper"), lambda t: t.name("kept")]).uniques([
                ["keeper"], ["kept"]]),
            lambda _: _.Kept.name("wts.Kept").ref(True).adjacencies([lambda t: t.relation([_.Keeps]).me("kept").owner(True)]),
        ])]))])


store.transaction(kept)
Loose = store.registered("wts.Loose")
with raises(ValueError, match="a 'wts.Kept' cannot be owned by a counted 'wts.Loose': nothing is owned by a counted object"):
    with store.transaction():
        store.wts.Loose(lambda l: l.keeps(lambda a: a.insert(lambda e: e.kept(lambda k: k))))
with raises(ValueError, match="nothing is owned by a counted object"), store.transaction():  # refused within another
    store.wts.Loose(lambda l: l.keeps(lambda a: a.insert(lambda e: e.kept(lambda k: k))))
with store.transaction():
    loose = store.wts.Loose(lambda l: l)
assert loose.as_schema() is Loose  # a refused write leaves nothing pending for the next
assert Validators.Validate(store)(Loose, loose) == [  # and validation refuses the schemas that would allow it
    "schema 'wts.Loose': adjacency 'keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is owned "
    "by a counted object, which a 'wts.Loose' is"]


def stray(txn):
    """wts.Stray, counted, which would own a wts.Kept through an anonymous adjacency."""
    return txn.uid("stray/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Stray": txn.Schemas.OfObject})
        .spec([lambda _: _.Stray.name("wts.Stray").ref(True).adjacencies([
            lambda t: t.relation([txn.wts.Keeps]).me("keeper")])])]))])


store.transaction(stray)
Stray = store.registered("wts.Stray")
with store.transaction():
    assert Validators.Validate(store)(Stray, store.wts.Stray(lambda s: s)) == [  # an anonymous adjacency, by its relation
        "schema 'wts.Stray': adjacency 'wts.Keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is "
        "owned by a counted object, which a 'wts.Stray' is"]
assert Validators.Validate(store)(Directory, features) == []  # a singleton owns

## IDN-06 · Lookup by keys: on any adjacency; `store.Instances` finds only singletons, and a counted object has no path

In [ ]:
def counted(txn):
    """wts.Free, with no owner() adjacency, and not a singleton: counted."""
    return txn.uid("free/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Free": txn.Schemas.OfObject})
        .spec([lambda _: _.Free.name("wts.Free").ref(True)])]))])


store.transaction(counted)
with store.transaction():
    free = store.wts.Free(lambda f: f)  # no owner() adjacency, and not a singleton: counted
with raises(LookupError, match="a 'wts.Free' is counted: it has no path"):
    Paths.of(free)
assert repr(store.Instances.wts) == "<Instances.wts>"
with raises(LookupError, match="no singleton named 'wts.Free': an owned object is found through its owners' adjacencies, "
                               "a counted one only through what links it"):
    store.Instances.wts.Free({"name": "free-1"})
with raises(LookupError, match="no singleton named 'wts.Definition'"):
    store.Instances.wts.Definition({"name": "DEF-693"})  # owned through Defines: found as features.dir({"name": ...})
with raises(AttributeError, match="_hidden"):
    store.Instances._hidden
with raises(LookupError, match="no entry of 'dir' has the keys {'name': 'DEF-999'}"):
    features.dir({"name": "DEF-999"})
with raises(KeyError, match="'nope' is not a property of 'wts.Defines'"):
    features.dir({"nope": 1})


def trio(txn):
    """wts.Trio, a relation of three links, through which wts.Features has `trio`."""
    S = txn.Schemas
    return txn.uid("trio/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Trio": S.OfRelation, "Directory": S.OfObject})
        .spec([
            lambda _: _.Trio.name("wts.Trio").links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]),
            lambda _: _.Directory.name("wts.Directory").adjacencies([lambda t: t.name("trio").relation([_.Trio]).me("a")]),
        ])]))])


store.transaction(trio)
with raises(TypeError, match="a lookup through 'trio' needs a relation of two links"):
    features.trio({})

## IDN-07 · Owned objects in documents and snapshots: a document refers to what it does not hold by its path; an owned root, placed by the entry that owns it; a snapshot carries them, each found again by the entry that owns it

In [ ]:
source, target = B.Store(), B.Store()
source.transaction(wts)
target.transaction(wts)
features = source.Instances.wts.Features
with source.transaction():  # wts.Features, defining DEF-001, which holds the clause C-1
    source.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.name("DEF-001").defn(lambda d: (
        d
        .title("Keys")
        .clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x)))
    )))))
Directory, Definition, Clause = (source.registered(f"wts.{name}") for name in ("Directory", "Definition", "Clause"))
graph = Plain.ToPlain(source).Reachable(Directory, features)
at = {"singleton": "wts.Features", "next": {"entry": {"i": "dir", "o": "def", "properties": {"name": "DEF-001"}, "next": {
    "target": {}}}}}
assert view(graph)["objects"] == [listed("wts.Directory", 0, {"dir": [{"defn": {"$ref": at}, "name": "DEF-001"}]})]  # alone
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph)  # what it owns is a snapshot's to carry
assert Plain.FromPlain(source).Reachable(Directory, graph) is features and len(features.dir) == 1  # held: found
defn = features.dir({"name": "DEF-001"})
written = Plain.ToPlain(source).OfObject(Definition, defn)  # an owned root: the entry that owns it, its owner by path
assert view(written)["objects"][0]["properties"]["def"] == [{"directory": {"$ref": {"singleton": "wts.Features", "next": {
    "target": {}}}}, "name": "DEF-001"}]
assert Plain.FromPlain(source).OfObject(Definition, written) is defn  # found by the entry that owns it
with raises(ValueError, match="a 'wts.Clause' is owned through an adjacency with no name: a document cannot give the entry "
                              "that owns it; name the adjacency"):
    Plain.ToPlain(source).OfObject(Clause, defn.clauses({"name": "C-1"}))
with source.transaction():
    source.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.name("DEF-003").defn(lambda d: d.title("Bare")))))
bare = json.loads(json.dumps(Plain.ToPlain(source).OfObject(Definition, features.dir({"name": "DEF-003"}))).replace("DEF-003", "DEF-004"))
made = Plain.FromPlain(source).OfObject(Definition, bare)  # an owned root none holds: made, placed by its owning entry
assert features.dir({"name": "DEF-004"}) is made and str(made.title) == "Bare"
renamed = json.loads(json.dumps(graph).replace('"o": "def"', '"o": "nope"'))
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    Plain.FromPlain(source).Reachable(Directory, renamed)  # a path through no owning adjacency names nothing

# A store's snapshot carries the owned objects: each a symbol, placed by the entry that owns it, its owner before it.
snapshot = Plain.ToPlain(source).Snapshot()
third = B.Store()
made = Plain.FromPlain(third).Snapshot(snapshot)
assert Plain.ToPlain(third).Snapshot() == snapshot
assert third.Instances.wts.Features.dir({"name": "DEF-001"}).clauses({"name": "C-1"}).as_schema() is third.registered("wts.Clause")
again = Plain.FromPlain(third).Snapshot(snapshot)  # read again: each found, a clause through its definition's entry (60)
assert len(again) == len(made) and all(a is m for a, m in zip(again, made))
assert [str(e.name) for e in third.Instances.wts.Features.dir] == ["DEF-001", "DEF-003", "DEF-004"]